# Seed Demo Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/00_seed_data.ipynb)

Populates CoordiNode with a **tech industry knowledge graph** you can explore
in notebooks 01–03.

**Graph contents:**
- 10 people (engineers, researchers, founders)
- 6 companies
- 8 technologies / research areas
- ~35 relationships (WORKS_AT, FOUNDED, KNOWS, RESEARCHES, INVENTED, ACQUIRED, USES, …)

All nodes carry a `demo=true` property so they can be bulk-deleted without
touching other data.

**Environments:**
- **Google Colab** — uses `coordinode-embedded` (in-process Rust engine, no server needed). First run compiles from source (~5 min); subsequent runs use the pip cache.
- **Local / Docker Compose** — connects to a running CoordiNode server via gRPC.

> **⚠️ Note for real-server use:** MERGE operations match nodes by name (e.g. `"Google"`). Run this notebook against a fresh/empty database or embedded mode to avoid accidentally tagging and then deleting pre-existing nodes during cleanup.

## Install dependencies

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# Always install coordinode-embedded so it's available as a local fallback.
if IN_COLAB:
    # Install Rust toolchain via rustup (https://rustup.rs).
    # Colab's apt packages ship rustc ≤1.75, which cannot build coordinode-embedded
    # (requires Rust ≥1.80 for maturin/pyo3). apt-get is not a viable alternative here.
    # Download the installer to a temp file and execute it explicitly — this avoids
    # piping remote content directly into a shell while maintaining HTTPS/TLS security
    # through Python's default ssl context (cert-verified, TLS 1.2+).
    import ssl as _ssl, tempfile as _tmp, urllib.request as _ur
    _ctx = _ssl.create_default_context()
    with _tmp.NamedTemporaryFile(mode="wb", suffix=".sh", delete=False) as _f:
        with _ur.urlopen("https://sh.rustup.rs", context=_ctx) as _r:
            _f.write(_r.read())
        _rustup_path = _f.name
    try:
        subprocess.run(["/bin/sh", _rustup_path, "-s", "--", "-y", "-q"], check=True)
    finally:
        os.unlink(_rustup_path)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "coordinode",
        "git+https://github.com/structured-world/coordinode-python.git#subdirectory=coordinode-embedded",
        "nest_asyncio",
    ],
    check=True,
)

import nest_asyncio

nest_asyncio.apply()

print("Ready")

## Connect to CoordiNode

- **Colab**: uses `LocalClient(":memory:")` — in-process embedded engine, no server required.
- **Local with server**: connects to an existing CoordiNode on port 7080 (set `COORDINODE_ADDR` to override).
- **Local without server**: falls back to embedded engine automatically.

In [ ]:
import os, socket


def _port_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False


GRPC_PORT = int(os.environ.get("COORDINODE_PORT", "7080"))

if os.environ.get("COORDINODE_ADDR"):
    COORDINODE_ADDR = os.environ["COORDINODE_ADDR"]
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")
elif _port_open(GRPC_PORT):
    COORDINODE_ADDR = f"localhost:{GRPC_PORT}"
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")
else:
    # No server available — use the embedded in-process engine.
    from coordinode_embedded import LocalClient

    client = LocalClient(":memory:")
    print("Using embedded LocalClient (in-process)")

## Step 1 — Clear previous demo data

In [ ]:
result = client.cypher("MATCH (n {demo: true}) DETACH DELETE n")
print("Previous demo data removed")

## Step 2 — Create nodes

In [ ]:
# ── People ────────────────────────────────────────────────────────────────
people = [
    {"name": "Alice Chen", "role": "ML Researcher", "org": "DeepMind", "field": "Reinforcement Learning"},
    {"name": "Bob Torres", "role": "Staff Engineer", "org": "Google", "field": "Distributed Systems"},
    {"name": "Carol Smith", "role": "Founder & CEO", "org": "Synthex", "field": "NLP"},
    {"name": "David Park", "role": "Research Scientist", "org": "OpenAI", "field": "LLMs"},
    {"name": "Eva Müller", "role": "Systems Architect", "org": "Synthex", "field": "Graph Databases"},
    {"name": "Frank Liu", "role": "Principal Engineer", "org": "Meta", "field": "Graph ML"},
    {"name": "Grace Okafor", "role": "PhD Researcher", "org": "MIT", "field": "Knowledge Graphs"},
    {"name": "Henry Rossi", "role": "CTO", "org": "Synthex", "field": "Databases"},
    {"name": "Isla Nakamura", "role": "Senior Researcher", "org": "DeepMind", "field": "Graph Neural Networks"},
    {"name": "James Wright", "role": "Engineering Lead", "org": "Google", "field": "Search"},
]

for p in people:
    client.cypher("MERGE (n:Person {name: $name}) SET n.role = $role, n.field = $field, n.demo = true", params=p)

print(f"Created {len(people)} people")

# ── Companies ─────────────────────────────────────────────────────────────
companies = [
    {"name": "Google", "industry": "Technology", "founded": 1998, "hq": "Mountain View"},
    {"name": "Meta", "industry": "Technology", "founded": 2004, "hq": "Menlo Park"},
    {"name": "OpenAI", "industry": "AI Research", "founded": 2015, "hq": "San Francisco"},
    {"name": "DeepMind", "industry": "AI Research", "founded": 2010, "hq": "London"},
    {"name": "Synthex", "industry": "AI Startup", "founded": 2021, "hq": "Berlin"},
    {"name": "MIT", "industry": "Academia", "founded": 1861, "hq": "Cambridge"},
]

for c in companies:
    client.cypher(
        "MERGE (n:Company {name: $name}) SET n.industry = $industry, n.founded = $founded, n.hq = $hq, n.demo = true",
        params=c,
    )

print(f"Created {len(companies)} companies")

# ── Technologies ──────────────────────────────────────────────────────────
technologies = [
    {"name": "Transformer", "type": "Architecture", "year": 2017},
    {"name": "Graph Neural Network", "type": "Algorithm", "year": 2009},
    {"name": "Reinforcement Learning", "type": "Paradigm", "year": 1980},
    {"name": "Knowledge Graph", "type": "Data Model", "year": 2012},
    {"name": "Vector Database", "type": "Infrastructure", "year": 2019},
    {"name": "RAG", "type": "Technique", "year": 2020},
    {"name": "LLM", "type": "Model Class", "year": 2018},
    {"name": "GraphRAG", "type": "Technique", "year": 2023},
]

for t in technologies:
    client.cypher("MERGE (n:Technology {name: $name}) SET n.type = $type, n.year = $year, n.demo = true", params=t)

print(f"Created {len(technologies)} technologies")

## Step 3 — Create relationships

In [ ]:
edges = [
    # WORKS_AT
    ("Alice Chen", "WORKS_AT", "DeepMind", {}),
    ("Bob Torres", "WORKS_AT", "Google", {}),
    ("Carol Smith", "WORKS_AT", "Synthex", {"since": 2021}),
    ("David Park", "WORKS_AT", "OpenAI", {}),
    ("Eva Müller", "WORKS_AT", "Synthex", {"since": 2022}),
    ("Frank Liu", "WORKS_AT", "Meta", {}),
    ("Grace Okafor", "WORKS_AT", "MIT", {}),
    ("Henry Rossi", "WORKS_AT", "Synthex", {"since": 2021}),
    ("Isla Nakamura", "WORKS_AT", "DeepMind", {}),
    ("James Wright", "WORKS_AT", "Google", {}),
    # FOUNDED
    ("Carol Smith", "FOUNDED", "Synthex", {"year": 2021}),
    ("Henry Rossi", "CO_FOUNDED", "Synthex", {"year": 2021}),
    # KNOWS
    ("Alice Chen", "KNOWS", "Isla Nakamura", {}),
    ("Alice Chen", "KNOWS", "David Park", {}),
    ("Carol Smith", "KNOWS", "Bob Torres", {}),
    ("Grace Okafor", "KNOWS", "Alice Chen", {}),
    ("Frank Liu", "KNOWS", "James Wright", {}),
    ("Eva Müller", "KNOWS", "Grace Okafor", {}),
    # RESEARCHES / WORKS_ON
    ("Alice Chen", "RESEARCHES", "Reinforcement Learning", {"since": 2019}),
    ("David Park", "RESEARCHES", "LLM", {"since": 2020}),
    ("Grace Okafor", "RESEARCHES", "Knowledge Graph", {"since": 2021}),
    ("Isla Nakamura", "RESEARCHES", "Graph Neural Network", {"since": 2020}),
    ("Frank Liu", "RESEARCHES", "Graph Neural Network", {}),
    ("Grace Okafor", "RESEARCHES", "GraphRAG", {"since": 2023}),
    # USES
    ("Synthex", "USES", "Knowledge Graph", {}),
    ("Synthex", "USES", "Vector Database", {}),
    ("Synthex", "USES", "RAG", {}),
    ("OpenAI", "USES", "Transformer", {}),
    ("Google", "USES", "Transformer", {}),
    # ACQUIRED
    ("Google", "ACQUIRED", "DeepMind", {"year": 2014}),
    # BUILDS_ON
    ("GraphRAG", "BUILDS_ON", "Knowledge Graph", {}),
    ("GraphRAG", "BUILDS_ON", "RAG", {}),
    ("RAG", "BUILDS_ON", "Vector Database", {}),
    ("LLM", "BUILDS_ON", "Transformer", {}),
]

src_names = {p["name"] for p in people}
tech_names = {t["name"] for t in technologies}
company_names = {c["name"] for c in companies}


def _label(name):
    if name in src_names:
        return "Person"
    if name in tech_names:
        return "Technology"
    if name in company_names:
        return "Company"
    raise ValueError(f"Unknown edge endpoint: {name!r}")


for src, rel, dst, props in edges:
    src_label = _label(src)
    dst_label = _label(dst)
    set_clause = ", ".join(f"r.{k} = ${k}" for k in props) if props else ""
    set_part = f" SET {set_clause}" if set_clause else ""
    client.cypher(
        f"MATCH (a:{src_label} {{name: $src}}) "
        f"MATCH (b:{dst_label} {{name: $dst}}) "
        f"MERGE (a)-[r:{rel}]->(b)" + set_part,
        params={"src": src, "dst": dst, **props},
    )

print(f"Created {len(edges)} relationships")

## Step 4 — Verify

In [ ]:
from collections import Counter

print("Node counts:")
for label in ["Person", "Company", "Technology"]:
    rows = client.cypher(f"MATCH (n:{label} {{demo: true}}) RETURN count(n) AS count")
    print(f"  {label:15s} {rows[0]['count']}")

# Fetch all types and count in Python (avoids aggregation limitations)
rels = client.cypher("MATCH (a {demo: true})-[r]->(b) RETURN type(r) AS rel")
counts = Counter(r["rel"] for r in rels)
print("\nRelationship counts:")
for rel, cnt in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {rel:20s} {cnt}")

In [ ]:
print("=== Who works at Synthex? ===")
rows = client.cypher(
    "MATCH (p:Person)-[:WORKS_AT]->(c:Company {name: $co}) RETURN p.name AS name, p.role AS role",
    params={"co": "Synthex"},
)
for r in rows:
    print(f"  {r['name']} — {r['role']}")

print("\n=== What does Synthex use? ===")
rows = client.cypher(
    "MATCH (c:Company {name: $co})-[:USES]->(t:Technology) RETURN t.name AS name", params={"co": "Synthex"}
)
for r in rows:
    print(f"  {r['name']}")

print("\n=== GraphRAG dependency chain ===")
rows = client.cypher(
    "MATCH (t:Technology {name: $tech})-[:BUILDS_ON*1..3]->(dep) RETURN dep.name AS dependency",
    params={"tech": "GraphRAG"},
)
for r in rows:
    print(f"  → {r['dependency']}")

print("\n✓ Demo data ready — open notebooks 01, 02, 03 to explore!")
client.close()